# Past / Current Service IO（正式命名版）

本版已將沈同學的 Past / Current Service 重新整理成 **Statistics Service 一致命名格式**。

## 這版的核心原則

本版不再使用「對應表」作為主要整合方式，而是直接把 Past / Current Service 的 input / output 欄位改成 正式命名。

也就是直接使用：

- `stations`
- `station_id`
- `station_name_zh`
- `station_name_en`
- `state`
- `metrics`
- `process_parameters`

不再使用：

- `machines`
- `machine_id`
- `process_name`
- `Qc`
- `Ut`
- `para`
- `cycle_time`

## 整體流程

```text
Past Service
 ↓
Current Service
 ↓
Future Service
 ↓
Statistics Service
 ↓
UI(E) / UI(M)
 ↓
Ontology / Runtime TTL
```


## 補強版說明：加入噴漆實務三大類資料

本 notebook 是在「正式命名版」的基礎上，額外加入噴漆機台更實務的三大類欄位。

保留原本未補強版本的原因是：目前尚未取得更前面的實際資料來源，因此不能假設所有欄位現在都一定存在。 
補強版的用途是提供後續和組員討論、擴充 schema / ontology / UI 對接時的參考。

### 本版新增三大類

1. 製程背景：`batch_id`、`product_id`、`part_id`、`recipe_name`、`paint_type`、`paint_batch_id`
2. 噴漆設備：`nozzle_id`、`spray_gun_id`、`paint_pressure_bar`、`paint_flow_rate_ml_min`、`atomizing_air_pressure_bar`
3. 噴塗品質與環境：`chamber_temperature_c`、`chamber_humidity_pct`、`film_thickness_um`、`color_difference_delta_e`、`defect_count`、`defect_type`

### 注意

補強欄位目前使用 placeholder 或 null，代表欄位設計方向，不代表已取得真實資料。


## 1. 站別命名統一

三站統一使用以下 正式命名：

| station_id | station_name_zh | station_name_en |
|---|---|---|
| M1 | 底漆站 | Primer Station |
| M2 | 面漆站 | Topcoat Station |
| M3 | 金漆站 | Gold Paint Station |

> 若原始資料曾使用「色漆」，本版一律統一為「面漆站 / Topcoat Station」。


## 2. Past Service Input

Past Service 負責查詢一段歷史範圍內的站別資料。 
本版 input 已改成 風格命名，使用 `stations[]`，不再使用 `machines[]`。


In [ ]:
past_service_input = {
 "service_name": "past_service",
 "schema_version": "service-contract-compatible",
 "range_variable": {
 "type": "history_range",
 "description": "Past Service 查詢的歷史範圍"
 },
 "window": {
 "mode": "time_or_batch",
 "window_type": "2hour_or_10batch",
 "window_size": "defined_by_system"
 },
 "sample_method": "defined_by_service",
 "target": [
 "state",
 "metrics",
 "process_parameters",
 "process_context",
 "paint_equipment",
 "quality_and_environment"
 ],
 "stations": [
 {
 "station_id": "M1",
 "station_name_zh": "底漆站",
 "station_name_en": "Primer Station",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "primer",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>"
 }
 },
 {
 "station_id": "M2",
 "station_name_zh": "面漆站",
 "station_name_en": "Topcoat Station",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "topcoat",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>"
 }
 },
 {
 "station_id": "M3",
 "station_name_zh": "金漆站",
 "station_name_en": "Gold Paint Station",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "gold_paint",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>"
 }
 }
 ]
}

past_service_input


## 2.1 補強三大類欄位設計理由

### 第一類：製程背景 `process_context`

噴漆製程中，不同批次、產品、零件、配方與漆料類型，可能會有不同的壓力、流量、膜厚與品質標準。 
因此補上：

```json
"process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "<primer|topcoat|gold_paint>",
 "paint_batch_id": "<string|null>"
}
```

### 第二類：噴漆設備 `paint_equipment`

噴嘴與噴槍是噴塗品質與堵塞風險的重要來源。 
若未來要做 ontology，`Nozzle`、`SprayGun` 也需要有資料可以連接。

```json
"paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>",
 "nozzle_status": "<string|null>",
 "spray_gun_status": "<string|null>"
}
```

### 第三類：噴塗品質與環境 `quality_and_environment`

噴塗品質不應只用 `quality_score_pct` 表示。 
膜厚、色差、缺陷、腔體溫濕度等，都可能影響噴塗結果。

```json
"quality_and_environment": {
 "chamber_temperature_c": "<number|null>",
 "chamber_humidity_pct": "<number|null>",
 "film_thickness_um": "<number|null>",
 "color_difference_delta_e": "<number|null>",
 "defect_count": "<number|null>",
 "defect_type": "<string|null>"
}
```

這些欄位目前是補強方向，不代表已取得真實資料。


## 3. Past Service Output

Past Service output 已改成 一致命名。 
重點是把原本的 `Qc`、`Ut`、`para`、`cycle_time` 全部整理進：

- `metrics`
- `process_parameters`

### 欄位命名

| 原概念 | 統一欄位 |
|---|---|
| 品質 Qc | `metrics.quality_score_pct` |
| 利用率 Ut | `process_parameters.utilization_pct` |
| 壓力 | `metrics.pressure_bar` |
| 流量 | `metrics.flow_rate_ml_min` |
| 溫度 | `process_parameters.temperature_c` |
| Cycle time | `process_parameters.cycle_time_sec` |

> `flow_rate_ml_min` 這個欄位名稱先與 / UI 對齊；實際資料進入時，上游需確認或換算成 ml/min 後再填入。


In [ ]:
past_service_output = {
 "service_name": "past_service",
 "schema_version": "service-contract-compatible",
 "window": {
 "mode": "time_or_batch",
 "window_type": "2hour_or_10batch",
 "display_label": "past window"
 },
 "stations": [
 {
 "station_id": "M1",
 "station_name_zh": "底漆站",
 "station_name_en": "Primer Station",
 "state": "Running",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "<primer|topcoat|gold_paint>",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>",
 "nozzle_status": "<string|null>",
 "spray_gun_status": "<string|null>"
 },
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "paint_pressure_bar": "<number|null>",
 "paint_flow_rate_ml_min": "<number|null>",
 "atomizing_air_pressure_bar": "<number|null>",
 "air_flow_rate_value": "<number|null>",
 "air_flow_rate_unit": "<string|null>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 },
 "quality_and_environment": {
 "chamber_temperature_c": "<number|null>",
 "chamber_humidity_pct": "<number|null>",
 "film_thickness_um": "<number|null>",
 "color_difference_delta_e": "<number|null>",
 "defect_count": "<number|null>",
 "defect_type": "<string|null>"
 }
 },
 {
 "station_id": "M2",
 "station_name_zh": "面漆站",
 "station_name_en": "Topcoat Station",
 "state": "Running",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "<primer|topcoat|gold_paint>",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>",
 "nozzle_status": "<string|null>",
 "spray_gun_status": "<string|null>"
 },
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "paint_pressure_bar": "<number|null>",
 "paint_flow_rate_ml_min": "<number|null>",
 "atomizing_air_pressure_bar": "<number|null>",
 "air_flow_rate_value": "<number|null>",
 "air_flow_rate_unit": "<string|null>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 },
 "quality_and_environment": {
 "chamber_temperature_c": "<number|null>",
 "chamber_humidity_pct": "<number|null>",
 "film_thickness_um": "<number|null>",
 "color_difference_delta_e": "<number|null>",
 "defect_count": "<number|null>",
 "defect_type": "<string|null>"
 }
 },
 {
 "station_id": "M3",
 "station_name_zh": "金漆站",
 "station_name_en": "Gold Paint Station",
 "state": "Maintenance",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "<primer|topcoat|gold_paint>",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>",
 "nozzle_status": "<string|null>",
 "spray_gun_status": "<string|null>"
 },
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "paint_pressure_bar": "<number|null>",
 "paint_flow_rate_ml_min": "<number|null>",
 "atomizing_air_pressure_bar": "<number|null>",
 "air_flow_rate_value": "<number|null>",
 "air_flow_rate_unit": "<string|null>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 },
 "quality_and_environment": {
 "chamber_temperature_c": "<number|null>",
 "chamber_humidity_pct": "<number|null>",
 "film_thickness_um": "<number|null>",
 "color_difference_delta_e": "<number|null>",
 "defect_count": "<number|null>",
 "defect_type": "<string|null>"
 }
 }
 ],
 "notes": [
 "Past Service 主要提供歷史視窗內的站別狀態、metrics、process_parameters，以及可選的噴漆製程背景、設備、品質與環境欄位。",
 "availability_pct、clog_rate_pct、maintainability_pct、risk_text 若不是 Past Service 直接產生，可先保留 null，由後續 Statistics / Rule / Future Service 補齊。"
 ]
}

past_service_output


## 4. Current Service Input

Current Service 負責取得目前或近即時的站別資料。 
本版同樣使用 一致命名，不再使用 `machine_id` 或 `machines`。


In [ ]:
current_service_input = {
 "service_name": "current_service",
 "schema_version": "service-contract-compatible",
 "window": {
 "mode": "current",
 "window_type": "current_window",
 "window_size": "defined_by_system"
 },
 "sample_method": "defined_by_service",
 "target": [
 "state",
 "metrics",
 "process_parameters",
 "process_context",
 "paint_equipment",
 "quality_and_environment"
 ],
 "stations": [
 {
 "station_id": "M1",
 "station_name_zh": "底漆站",
 "station_name_en": "Primer Station",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "primer",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>"
 }
 },
 {
 "station_id": "M2",
 "station_name_zh": "面漆站",
 "station_name_en": "Topcoat Station",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "topcoat",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>"
 }
 },
 {
 "station_id": "M3",
 "station_name_zh": "金漆站",
 "station_name_en": "Gold Paint Station",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "gold_paint",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>"
 }
 }
 ]
}

current_service_input


## 5. Current Service Output

Current Service output 也已改成 一致命名。 
此處的資料可直接作為 Statistics Service 的上游輸入之一。

> 本版不再另外做 mapping table，因為欄位名稱本身已經對齊 。


In [ ]:
current_service_output = {
 "service_name": "current_service",
 "schema_version": "service-contract-compatible",
 "window": {
 "mode": "current",
 "window_type": "current_window",
 "display_label": "current"
 },
 "stations": [
 {
 "station_id": "M1",
 "station_name_zh": "底漆站",
 "station_name_en": "Primer Station",
 "state": "Running",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "<primer|topcoat|gold_paint>",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>",
 "nozzle_status": "<string|null>",
 "spray_gun_status": "<string|null>"
 },
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "paint_pressure_bar": "<number|null>",
 "paint_flow_rate_ml_min": "<number|null>",
 "atomizing_air_pressure_bar": "<number|null>",
 "air_flow_rate_value": "<number|null>",
 "air_flow_rate_unit": "<string|null>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 },
 "quality_and_environment": {
 "chamber_temperature_c": "<number|null>",
 "chamber_humidity_pct": "<number|null>",
 "film_thickness_um": "<number|null>",
 "color_difference_delta_e": "<number|null>",
 "defect_count": "<number|null>",
 "defect_type": "<string|null>"
 }
 },
 {
 "station_id": "M2",
 "station_name_zh": "面漆站",
 "station_name_en": "Topcoat Station",
 "state": "Running",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "<primer|topcoat|gold_paint>",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>",
 "nozzle_status": "<string|null>",
 "spray_gun_status": "<string|null>"
 },
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "paint_pressure_bar": "<number|null>",
 "paint_flow_rate_ml_min": "<number|null>",
 "atomizing_air_pressure_bar": "<number|null>",
 "air_flow_rate_value": "<number|null>",
 "air_flow_rate_unit": "<string|null>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 },
 "quality_and_environment": {
 "chamber_temperature_c": "<number|null>",
 "chamber_humidity_pct": "<number|null>",
 "film_thickness_um": "<number|null>",
 "color_difference_delta_e": "<number|null>",
 "defect_count": "<number|null>",
 "defect_type": "<string|null>"
 }
 },
 {
 "station_id": "M3",
 "station_name_zh": "金漆站",
 "station_name_en": "Gold Paint Station",
 "state": "Maintenance",
 "process_context": {
 "batch_id": "<string|null>",
 "product_id": "<string|null>",
 "part_id": "<string|null>",
 "recipe_name": "<string|null>",
 "paint_type": "<primer|topcoat|gold_paint>",
 "paint_batch_id": "<string|null>"
 },
 "paint_equipment": {
 "nozzle_id": "<string|null>",
 "spray_gun_id": "<string|null>",
 "nozzle_status": "<string|null>",
 "spray_gun_status": "<string|null>"
 },
 "metrics": {
 "pressure_bar": "<number>",
 "flow_rate_ml_min": "<number>",
 "paint_pressure_bar": "<number|null>",
 "paint_flow_rate_ml_min": "<number|null>",
 "atomizing_air_pressure_bar": "<number|null>",
 "air_flow_rate_value": "<number|null>",
 "air_flow_rate_unit": "<string|null>",
 "quality_score_pct": "<number>",
 "availability_pct": "<number|null>",
 "clog_rate_pct": "<number|null>",
 "maintainability_pct": "<number|null>",
 "risk_text": "<string|null>"
 },
 "process_parameters": {
 "temperature_c": "<number>",
 "utilization_pct": "<number>",
 "cycle_time_sec": "<number>"
 },
 "quality_and_environment": {
 "chamber_temperature_c": "<number|null>",
 "chamber_humidity_pct": "<number|null>",
 "film_thickness_um": "<number|null>",
 "color_difference_delta_e": "<number|null>",
 "defect_count": "<number|null>",
 "defect_type": "<string|null>"
 }
 }
 ],
 "notes": [
 "Current Service 提供目前站別狀態、metrics、process_parameters，以及可選的噴漆製程背景、設備、品質與環境欄位。",
 "若某些風險或維護指標不是 Current Service 直接產生，可先保留 null，交由後續 Statistics / Rule / Future Service 補齊。"
 ]
}

current_service_output


## 6. Past + Current 給 Statistics Service 的整合輸出

因為 Past / Current 現在已經直接使用 一致命名，所以這裡不需要額外 mapping。 
Statistics Service 可以直接讀取：

```text
past_current_integrated_output.past.stations[]
past_current_integrated_output.current.stations[]
```


In [ ]:
past_current_integrated_output = {
 "schema_version": "service-contract-compatible",
 "description": "Past / Current Service 使用 一致命名後，提供給 Statistics Service 的整合輸入",
 "past": past_service_output,
 "current": current_service_output,
 "next_service": {
 "statistics_service": {
 "expected_input": [
 "stations[].station_id",
 "stations[].station_name_zh",
 "stations[].station_name_en",
 "stations[].state",
 "stations[].metrics",
 "stations[].process_parameters",
 "stations[].process_context",
 "stations[].paint_equipment",
 "stations[].quality_and_environment"
 ],
 "to_be_completed_by_statistics_or_rule_service": [
 "summary",
 "component_overview",
 "trend_series",
 "availability_pct",
 "clog_rate_pct",
 "maintainability_pct",
 "risk_text"
 ]
 },
 "future_service": {
 "expected_input": [
 "past.stations[]",
 "current.stations[]",
 "target",
 "range",
 "confidence_level"
 ],
 "note": "Future Service 尚未定案，此處只保留可能需要的輸入方向。"
 }
 }
}

past_current_integrated_output


## 7. 本版最終整理

### Past Service

```text
Input:
stations[]
window
sample_method
target

Output:
stations[]
state
metrics
process_parameters
```

### Current Service

```text
Input:
stations[]
window
sample_method
target

Output:
stations[]
state
metrics
process_parameters
```

### 與 對齊後的好處

1. 不需要再另外用對應表把 `machine_id` 改成 `station_id`。
2. 不需要再把 `Qc` 改成 `quality_score_pct`，因為一開始就用 正式命名。
3. 不需要再把 `Ut` 改成 `utilization_pct`，因為一開始就用 正式命名。
4. Past / Current 可以直接被 Statistics Service 讀取。
5. UI(E) 和 ontology 後續接資料時，欄位名稱不會再打架。

### 最終流程

```text
Past Service
 ↓
Current Service
 ↓
Statistics Service
 ↓
UI(E) / UI(M)
 ↓
Ontology / Runtime TTL
```
